# CDIL-CNN Model (Wisconsin Breast Cancer Dataset)

### Clone Repo

In [ ]:
!git clone https://github.com/LeiCheng-no/CDIL-CNN.git

In [ ]:
!ls CDIL-CNN

### Imports

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    cohen_kappa_score,
    recall_score,
    precision_score,
    roc_auc_score
)

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input, Conv1D, Dense, Dropout, GlobalAveragePooling1D,
    BatchNormalization, Activation, Add
)
from tensorflow.keras.callbacks import EarlyStopping

### Load and Split Breast Cancer Data

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42
)
# 60% train, 20% val, 20% test

### Scale and Reshape

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

n_features = X_train_scaled.shape[1]

X_train_seq = X_train_scaled.reshape((X_train_scaled.shape[0], n_features, 1))
X_val_seq   = X_val_scaled.reshape((X_val_scaled.shape[0], n_features, 1))
X_test_seq  = X_test_scaled.reshape((X_test_scaled.shape[0], n_features, 1))

print(X_train_seq.shape, X_val_seq.shape, X_test_seq.shape)

### Define a simple CDIL-style residual block

In [ ]:
def dilated_residual_block(x, filters, dilation_rate):
    shortcut = x

    x = Conv1D(filters, kernel_size=3, padding="same", dilation_rate=dilation_rate)(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv1D(filters, kernel_size=3, padding="same", dilation_rate=dilation_rate)(x)
    x = BatchNormalization()(x)

    # match shortcut channels if needed
    if shortcut.shape[-1] != filters:
        shortcut = Conv1D(filters, kernel_size=1, padding="same")(shortcut)

    x = Add()([x, shortcut])
    x = Activation("relu")(x)
    return x

### Build the CDIL-CNN-style model

In [ ]:
tf.keras.utils.set_random_seed(42)

inputs = Input(shape=(n_features, 1))

x = Conv1D(32, kernel_size=3, padding="same", activation="relu")(inputs)

# progressively larger dilation rates
x = dilated_residual_block(x, filters=32, dilation_rate=1)
x = dilated_residual_block(x, filters=32, dilation_rate=2)
x = dilated_residual_block(x, filters=64, dilation_rate=4)
x = dilated_residual_block(x, filters=64, dilation_rate=8)

x = GlobalAveragePooling1D()(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation="sigmoid")(x)

cdil_cnn_model = Model(inputs, outputs)

cdil_cnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

cdil_cnn_model.summary()

### Train the model

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history = cdil_cnn_model.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=60,
    batch_size=32,
    verbose=1,
    callbacks=[early_stop]
)

### Evaluate the model

In [ ]:
train_loss, train_acc, train_auc = cdil_cnn_model.evaluate(X_train_seq, y_train, verbose=0)
val_loss, val_acc, val_auc = cdil_cnn_model.evaluate(X_val_seq, y_val, verbose=0)

y_test_prob = cdil_cnn_model.predict(X_test_seq, verbose=0).ravel()
y_test_pred = (y_test_prob >= 0.5).astype(int)

test_acc = accuracy_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
kappa = cohen_kappa_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
auc = roc_auc_score(y_test, y_test_prob)

cdil_results = pd.DataFrame([{
    "Model": "CDIL-CNN",
    "Train_accuracy": round(train_acc, 4),
    "Val_accuracy": round(val_acc, 4),
    "Test_accuracy": round(test_acc, 4),
    "F1_score": round(f1, 4),
    "Kappa": round(kappa, 4),
    "Recall": round(recall, 4),
    "Precision": round(precision, 4),
    "AUC": round(auc, 4)
}])

print(cdil_results.to_string(index=False))